In [0]:
from databricks.sdk import WorkspaceClient

In [0]:
w = WorkspaceClient()

## Get Databricks Workspace URL Using Databricks SDK

In [0]:
workspace_url = w.config.host
print(workspace_url)

## Create a Personal Access Token Using Databricks SDK

In [0]:
# Create a token with 2 days lifetime (2 * 24 * 60 * 60 seconds)
lifetime_seconds = 2 * 24 * 60 * 60
new_token = w.tokens.create(comment = 'SDK Generated Token', lifetime_seconds = lifetime_seconds)
print(f'Token value: {new_token.token_value}')

## View All Personal Access Tokens of a Databricks Workspace Using Databricks SDK

In [0]:
# List All Valid Tokens Generated by the Current User in the Current Databricks Workspace
for token in w.tokens.list():
    print(f"ID: {token.token_id}, Comment: {token.comment}, Created: {token.creation_time}, Token value: {token.token_value}")

# List All Tokens Associated with the Current Databricks Workspace
all_tokens = w.token_management.list()

for token_info in all_tokens:
    print(f"User: {token_info.created_by_username}, ID: {token_info.token_id}, Token value: {token_info.token_value}, Expiry: {token_info.expiry_time}")

## Create a Secret Scope Using Databricks REST API

In [0]:
import requests
import json

# --------------------- CONFIGURATIONS ---------------------

DATABRICKS_INSTANCE = workspace_url
DATABRICKS_TOKEN = new_token.token_value

scope_name = 'oindrila_secret_scope' # Name of the Scope to Create
backend_type = 'DATABRICKS' # Use 'AZURE_KEYVAULT', if Integrating with Azure Key Vault

# --------------------- API ENDPOINTS ---------------------
api_url = f'{DATABRICKS_INSTANCE}/api/2.0/secrets/scopes/create'

api_header = {
    "Authorization" : f"Bearer {DATABRICKS_TOKEN}",
    "Content-Type" : "application/json"
}

payload = {
    "scope" : scope_name
}

# --------------------- SEND REQUEST ---------------------
response = requests.post(api_url, headers = api_header, data = json.dumps(payload))

# --------------------- HANDLE RESPONSE ---------------------
if response.status_code == 200:
    print(f"Secret Scope '{scope_name}' is Created Successfully")
else:
    print(f"Failed to Create Secret Scope '{scope_name}'")
    print("Status Code : ", response.status_code)
    print("Response : ", response.text)

## Create a Secret in Existing Secret Scope Using Databricks REST API

In [0]:
import requests
import json

# --------------------- CONFIGURATIONS ---------------------

scope_name = 'oindrila_secret_scope' # Name of the Existing Scope
secret_key = 'first_pat' # Name of the Secret to Create

# --------------------- API ENDPOINTS ---------------------
api_url = f'{DATABRICKS_INSTANCE}/api/2.0/secrets/put'

api_header = {
    "Authorization" : f"Bearer {DATABRICKS_TOKEN}",
    "Content-Type" : "application/json"
}

payload = {
    "scope" : scope_name,
    "key" : secret_key,
    "string_value" : DATABRICKS_TOKEN
}

# --------------------- SEND REQUEST ---------------------
response = requests.post(api_url, headers = api_header, data = json.dumps(payload))

# --------------------- HANDLE RESPONSE ---------------------
if response.status_code == 200:
    print(f"Secret '{secret_key}' is Created Successfully in Secret Scope '{scope_name}'")
else:
    print(f"Failed to Create Secret '{secret_key}' in Secret Scope '{scope_name}'")
    print("Status Code : ", response.status_code)
    print("Response : ", response.text)

## Display the Metadata of All the Secrets Present in an Existing Secret Scope Using Databricks REST API

In [0]:
import requests

# --------------------- CONFIGURATIONS ---------------------

scope_name = 'oindrila_secret_scope' # Name of the Existing Scope

# --------------------- API ENDPOINTS ---------------------
api_url = f'{DATABRICKS_INSTANCE}/api/2.0/secrets/list?scope={scope_name}'

api_header = {
    "Authorization" : f"Bearer {DATABRICKS_TOKEN}",
    "Content-Type" : "application/json"
}

# --------------------- SEND REQUEST ---------------------
response = requests.get(api_url, headers = api_header)

# --------------------- HANDLE RESPONSE ---------------------
if response.status_code == 400:
    print(f"Secret Scope '{scope_name}' Does Not Exists")
    exit()

secrets = response.json().get("secrets", [])
print(secrets)

## Display All the Secret Scopes Present in the Current Databricks Workspace Using "DBUtils"

In [0]:
dbutils.secrets.listScopes()

## Display the Metadata of All the Secrets Present in an Existing Secret Scope of the Current Databricks Workspace Using "DBUtils"

In [0]:
dbutils.secrets.list("oindrila_secret_scope")

## Display the Value of a Secret Present in an Existing Secret Scope of the Current Databricks Workspace Using "DBUtils"

In [0]:
pat_secret_key = dbutils.secrets.get(scope = "oindrila_secret_scope", key = "first_pat")
print(pat_secret_key)

In [0]:
for char in pat_secret_key:
    print(char)

## Delete All the Secrets Present in an Existing Secret Scope Using Databricks REST API

In [0]:
import requests

# --------------------- CONFIGURATIONS ---------------------

scope_name = 'oindrila_secret_scope' # Name of the Existing Scope

# --------------------- API ENDPOINTS ---------------------
api_url = f'{DATABRICKS_INSTANCE}/api/2.0/secrets/delete'

api_header = {
    "Authorization" : f"Bearer {DATABRICKS_TOKEN}",
    "Content-Type" : "application/json"
}

# --------------------- SEND REQUEST AND HANDLE RESPONSE ---------------------
for secret in list(dbutils.secrets.list("oindrila_secret_scope")):
    requests.post(api_url, headers = api_header, json = {"scope" : scope_name, "key" : secret.key})
    print(f"Secret '{secret.key}' is Successfully Deleted in the Secret Scope '{scope_name}'")

## Delete the Existing Secret Scope Using Databricks REST API

In [0]:
import requests

# --------------------- CONFIGURATIONS ---------------------

scope_name = 'oindrila_secret_scope' # Name of the Existing Scope to Delete

# --------------------- API ENDPOINTS ---------------------
api_url = f'{DATABRICKS_INSTANCE}/api/2.0/secrets/scopes/delete'

api_header = {
    "Authorization" : f"Bearer {DATABRICKS_TOKEN}",
    "Content-Type" : "application/json"
}

# --------------------- SEND REQUEST ---------------------
response = requests.post(api_url, headers = api_header, json = {"scope" : scope_name})

# --------------------- HANDLE RESPONSE ---------------------
if response.status_code == 200:
    print(f"Secret Scope '{scope_name}' Deleted Successfully")
else:
    print(f"Failed to Delete Secret Scope '{scope_name}'")
    print("Status Code : ", response.status_code)
    print("Response : ", response.text)